In [6]:
library(rsample)     # data splitting 
library(dplyr)       # data wrangling
library(rpart)       # performing regression trees
library(rpart.plot)  # plotting regression trees
library(ipred)       # bagging
library(caret)       # bagging
library(tidyverse)

Warning message:
"Paket 'purrr' wurde unter R Version 4.4.3 erstellt"
Warning message:
"Paket 'forcats' wurde unter R Version 4.4.1 erstellt"
-- Attaching core tidyverse packages ------------------------ tidyverse 2.0.0 --
v forcats   1.0.1     v stringr   1.5.1
v lubridate 1.9.3     v tibble    3.2.1
v purrr     1.2.1     v tidyr     1.3.1
v readr     2.1.5     
-- Conflicts ------------------------------------------ tidyverse_conflicts() --
x dplyr::filter() masks stats::filter()
x dplyr::lag()    masks stats::lag()
x purrr::lift()   masks caret::lift()
i Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [28]:
set.seed(123)
merged <- read_csv("data/merged_data.csv")
ames_split <- initial_split(merged, prop = 0.75)
ames_train <- training(ames_split)
ames_test  <- testing(ames_split)

Rows: 254 Columns: 404
-- Column specification --------------------------------------------------------
Delimiter: ","
chr   (1): County
dbl (380): cve, outbreak, enrollment, PHR, pct_hispanic, pct_black, pct_whit...
lgl  (23): median_income, Advised to Cut Down Salt - Do not use salt, Diabet...

i Use `spec()` to retrieve the full column specification for this data.
i Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [ ]:
library(rpart)

# 1. Drop non-numeric / ID columns
merged_model <- merged[, !names(merged) %in% c("County")]

# 2. Sanitize column names — this is the key fix
names(merged_model) <- make.names(names(merged_model), unique = TRUE)

# 3. Fit the tree
fit <- rpart(outbreak ~ ., data = merged_model, method = "anova")



In [11]:
fit

n= 254 

node), split, n, deviance, yval
      * denotes terminal node

1) root 254 181165.60  2.925197  
  2) pct_uninsured< 27.9 247  11506.09  1.323887 *
  3) pct_uninsured>=27.9 7 146677.70 59.428570 *

In [32]:
set.seed(123)
merged <- read_csv("data/merged_data.csv")
ames_split <- initial_split(merged, prop = 0.75)
ames_train <- training(ames_split)
ames_test  <- testing(ames_split)

Rows: 254 Columns: 404
-- Column specification --------------------------------------------------------
Delimiter: ","
chr   (1): County
dbl (380): cve, outbreak, enrollment, PHR, pct_hispanic, pct_black, pct_whit...
lgl  (23): median_income, Advised to Cut Down Salt - Do not use salt, Diabet...

i Use `spec()` to retrieve the full column specification for this data.
i Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [33]:
library(rpart)

# 1. Drop non-numeric / ID columns
merged_model <- merged[, !names(merged) %in% c("County")]

# 2. Sanitize column names — this is the key fix
names(merged_model) <- make.names(names(merged_model), unique = TRUE)

# 3. Fit the tree
fit <- rpart(outbreak ~ ., data = merged_model, method = "anova", control = rpart.control(minsplit = 10, cp = 0.005))
fit

n= 254 

node), split, n, deviance, yval
      * denotes terminal node

 1) root 254 181165.6000   2.9251970  
   2) pct_uninsured< 30.2 251  11511.7600   1.3107570  
     4) Pap...HPV.Test.Past.5.Yrs.21...No< 64.15 204   1086.8380   0.3970588 *
     5) Pap...HPV.Test.Past.5.Yrs.21...No>=64.15 47   9515.4040   5.2765960  
      10) enrollment< 22145 44   4217.6360   3.0909090  
        20) pct_uninsured< 26.25 41    750.1951   1.4634150 *
        21) pct_uninsured>=26.25 3   1874.6670  25.3333300 *
      11) enrollment>=22145 3   2004.6670  37.3333300 *
   3) pct_uninsured>=30.2 3 114264.0000 138.0000000 *

In [34]:
hyper_grid <- expand.grid(
  minsplit = seq(5, 20, 1),
  maxdepth = seq(8, 15, 1)
)

head(hyper_grid)

nrow(hyper_grid)

,minsplit,maxdepth
,<dbl>,<dbl>
1,5,8
2,6,8
3,7,8
4,8,8
5,9,8
6,10,8


[1] 128

In [35]:

models <- list()

for (i in 1:nrow(hyper_grid)) {
  
  # get minsplit, maxdepth values at row i
  minsplit <- hyper_grid$minsplit[i]
  maxdepth <- hyper_grid$maxdepth[i]

  # train a model and store in the list
  models[[i]] <- rpart(
    formula = outbreak ~ .,
    data    = merged_model,
    method  = "anova",
    control = list(minsplit = minsplit, maxdepth = maxdepth)
    )
}

In [36]:
get_cp <- function(x) {
  min    <- which.min(x$cptable[, "xerror"])
  cp <- x$cptable[min, "CP"] 
}

# function to get minimum error
get_min_error <- function(x) {
  min    <- which.min(x$cptable[, "xerror"])
  xerror <- x$cptable[min, "xerror"] 
}

hyper_grid %>%
  mutate(
    cp    = purrr::map_dbl(models, get_cp),
    error = purrr::map_dbl(models, get_min_error)
    ) %>%
  arrange(error) %>%
  top_n(-5, wt = error)

minsplit,maxdepth,cp,error
<dbl>,<dbl>,<dbl>,<dbl>
8,10,0.1268551,1.006742
17,14,0.1268551,1.006776
7,9,0.1268551,1.006884
16,14,0.1268551,1.006967
13,15,0.1268551,1.006998


In [37]:
optimal_tree <- rpart(
    formula = outbreak ~ .,
    data    = merged_model,
    method  = "anova",
    control = list(minsplit = 8, maxdepth = 10, cp = 0.1268551)
    )
optimal_tree
pred <- predict(optimal_tree, newdata = merged_model)
RMSE(pred = pred, obs = merged_model$outbreak)
pred

n= 254 

node), split, n, deviance, yval
      * denotes terminal node

1) root 254 181165.6 2.925197 *

[1] 26.70675

[1] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
  [9] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
 [17] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
 [25] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
 [33] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
 [41] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
 [49] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
 [57] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
 [65] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
 [73] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
 [81] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
 [89] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
 [97] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
[105] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
[113] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
[121] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
[129] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
[137] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
[145] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
[153] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
[161] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
[169] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
[177] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
[185] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
[193] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
[201] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
[209] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
[217] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
[225] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
[233] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
[241] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197
[249] 2.925197 2.925197 2.925197 2.925197 2.925197 2.925197